# LSTM 对照实验（LSTM_Funning）

使用与 LoRA 实验一致的数据划分与超参数，输出统一指标格式。

In [1]:
"""
第一部分：导入依赖
"""

import time
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from Bert_Config import CONFIG, setup_seed, load_raw_data, split_data

print("✅ 所有库导入完成")


"""
第二部分：统一配置
"""

MODEL_NAME = "LSTM"
DATA_PATH = CONFIG["DATA_DIR"]
RANDOM_SEED = CONFIG["RANDOM_SEED"]

MAX_VOCAB = CONFIG["CLASSIC_MAX_FEATURES"]
MAX_LEN = CONFIG["CLASSIC_MAX_LENGTH_TOKENS"]
EMBED_DIM = CONFIG["CLASSIC_EMBED_DIM"]
HIDDEN_DIM = CONFIG["CLASSIC_HIDDEN_DIM"]
EPOCHS = CONFIG["CLASSIC_EPOCHS"]
BATCH_SIZE = CONFIG["CLASSIC_BATCH_SIZE"]
LEARNING_RATE = CONFIG["CLASSIC_LEARNING_RATE"]

print("✅ 配置加载完成")


"""
第三部分：文本编码与数据集
"""


def build_vocab(texts, max_vocab):
    counter = Counter()
    for text in texts:
        counter.update(list(text))
    most_common = counter.most_common(max_vocab - 2)
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for idx, (tok, _) in enumerate(most_common, start=2):
        vocab[tok] = idx
    return vocab


def encode_text(text, vocab, max_len):
    tokens = list(text)
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens[:max_len]]
    if len(ids) < max_len:
        ids.extend([vocab["<PAD>"]] * (max_len - len(ids)))
    return ids


class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        input_ids = encode_text(self.texts[idx], self.vocab, self.max_len)
        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(
            self.labels[idx], dtype=torch.long
        )

print("✅ 数据集构建函数定义完成")


"""
第四部分：模型定义
"""


class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        out = self.dropout(hidden[-1])
        return self.fc(out)

print("✅ 模型定义完成")


"""
第五部分：评估函数
"""


def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            total_loss += loss.item() * batch_y.size(0)
            preds = logits.argmax(dim=1)
            all_labels.extend(batch_y.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())

    avg_loss = total_loss / len(dataloader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )
    return acc, precision, recall, f1, avg_loss

print("✅ 评估函数定义完成")


"""
第六部分：准备数据
"""

setup_seed(RANDOM_SEED)
df = load_raw_data(DATA_PATH)
train_df, val_df, test_df = split_data(df, RANDOM_SEED)

vocab = build_vocab(train_df["review"], MAX_VOCAB)

train_dataset = TextDataset(
    train_df["review"].tolist(),
    train_df["label"].astype(int).tolist(),
    vocab,
    MAX_LEN,
)
val_dataset = TextDataset(
    val_df["review"].tolist(),
    val_df["label"].astype(int).tolist(),
    vocab,
    MAX_LEN,
)
test_dataset = TextDataset(
    test_df["review"].tolist(),
    test_df["label"].astype(int).tolist(),
    vocab,
    MAX_LEN,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print("✅ 数据准备完成")


"""
第七部分：训练与评估
"""

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMClassifier(len(vocab), EMBED_DIM, HIDDEN_DIM).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

start = time.time()
for _ in range(EPOCHS):
    model.train()
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
elapsed = time.time() - start

train_metrics = evaluate(model, train_loader, criterion, device)
val_metrics = evaluate(model, val_loader, criterion, device)
test_metrics = evaluate(model, test_loader, criterion, device)

elapsed_min = elapsed / 60
print("\n" + "=" * 50)
print(f"模型: {MODEL_NAME}")
print(
    f"训练集 - Acc: {train_metrics[0]:.3f} | Precision: {train_metrics[1]:.3f} | "
    f"Recall: {train_metrics[2]:.3f} | F1: {train_metrics[3]:.3f} | "
    f"Loss: {train_metrics[4]:.3f}"
)
print(
    f"验证集 - Acc: {val_metrics[0]:.3f} | Precision: {val_metrics[1]:.3f} | "
    f"Recall: {val_metrics[2]:.3f} | F1: {val_metrics[3]:.3f} | "
    f"Loss: {val_metrics[4]:.3f}"
)
print(
    f"测试集 - Acc: {test_metrics[0]:.3f} | Precision: {test_metrics[1]:.3f} | "
    f"Recall: {test_metrics[2]:.3f} | F1: {test_metrics[3]:.3f} | "
    f"Loss: {test_metrics[4]:.3f}"
)
print(f"训练时间: {elapsed:.1f} 秒 ({elapsed_min:.2f} 分钟)")
print("=" * 50)


C:\Users\19836\miniconda3\envs\py8\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 所有库导入完成
✅ 配置加载完成
✅ 数据集构建函数定义完成
✅ 模型定义完成
✅ 评估函数定义完成
✅ 数据准备完成

模型: LSTM
训练集 - Acc: 0.856 | Precision: 0.760 | Recall: 0.830 | F1: 0.793 | Loss: 0.356
验证集 - Acc: 0.836 | Precision: 0.740 | Recall: 0.782 | F1: 0.761 | Loss: 0.390
测试集 - Acc: 0.834 | Precision: 0.740 | Recall: 0.775 | F1: 0.757 | Loss: 0.400
训练时间: 143.7 秒 (2.40 分钟)
